# SwiGLU MLP: Math, SiLU Derivation, Gating & CuTeDSL

## 1. Where the MLP Block Sits in Inference

After the attention sublayer updates the hidden state, the MLP (also called the FFN, or Feed-Forward Network) processes each token independently. Unlike attention — where every token "looks at" every other token — the MLP applies the same transformation to each token's vector in isolation. No communication between tokens happens here.

```
x ∈ ℝ^{B×T×4096}   (residual stream after attention)
  │
  ▼  RMSNorm (ln2)    ← normalize before the MLP
  │
  ├─ gate = x @ W_gate.T    GEMM  [B·T, 4096] → [B·T, 12288]
  ├─ up   = x @ W_up.T      GEMM  [B·T, 4096] → [B·T, 12288]
  │
  ▼  SwiGLU: h = SiLU(gate) ⊙ up     element-wise  → [B·T, 12288]
  │   (gate decides what passes through; up provides what to pass)
  │
  ▼  down = h @ W_down.T    GEMM  [B·T, 12288] → [B·T, 4096]
  │
  ▼  residual: x = x_before_ln2 + down   → [B·T, 4096]
```

The 3 weight matrices (gate, up, down) together account for roughly **65% of all Qwen3-8B weights**: 3 × 12,288 × 4,096 × 2 bytes × 36 layers = **10.8 GB** out of the model's ~16 GB total.

## 2. I/O Specification

| Field | Value |
|---|---|
| Input | x_norm ∈ ℝ^{B×T×D} — output of ln2 (the pre-FFN RMSNorm) |
| Input dtype | bfloat16 |
| Input shape at decode | [1, 1, 4096] |
| Input shape at prefill | [B, T, 4096] |
| What produced input | RMSNorm of residual stream after attention sublayer |
| Intermediate | gate, up ∈ ℝ^{B×T×H_int} where H_int=12288 |
| h (after gating) | SiLU(gate) ⊙ up ∈ ℝ^{B×T×12288} — the "gated activations" |
| Output | down ∈ ℝ^{B×T×4096} — back to hidden dimension |
| Output dtype | bfloat16 |
| What consumes output | Residual add: x = x + down; then next layer's ln1 |
| Computation type | 2× large GEMM (gate, up) → element-wise gating → 1× large GEMM (down) |

## 3. The SwiGLU Formula

The three-matrix MLP with SwiGLU activation:
$$\text{SwiGLU}(x, W_\text{gate}, W_\text{up}, W_\text{down}) = \Big(\text{SiLU}(x W_\text{gate}^T) \odot (x W_\text{up}^T)\Big) W_\text{down}^T$$

where $\odot$ is element-wise multiplication and SiLU is:
$$\text{SiLU}(z) = z \cdot \sigma(z) = \frac{z}{1 + e^{-z}}$$

Expanding in full:
$$g_i = \sum_{k} x_k (W_\text{gate})_{i,k} \quad \text{(gate projection, dim } H_\text{int})$$
$$u_i = \sum_{k} x_k (W_\text{up})_{i,k} \quad \text{(up projection, dim } H_\text{int})$$
$$h_i = \text{SiLU}(g_i) \cdot u_i = \frac{g_i}{1 + e^{-g_i}} \cdot u_i$$
$$y_j = \sum_{i} h_i (W_\text{down})_{j,i} \quad \text{(down projection, dim } D)$$

## 4. Derivation of SiLU (Sigmoid Linear Unit)

SiLU is derived from GLU (Gating Linear Unit, Dauphin et al. 2017) and Swish (Ramachandran et al. 2017):

**GLU baseline:**
$$\text{GLU}(x, W, V) = (xW) \odot \sigma(xV)$$
Here σ(xV) is the "gate" — values near 0 block, near 1 pass.

**Swish:** Replace the hard sigmoid gate with a smooth self-gate:
$$\text{Swish}(z) = z \cdot \sigma(z)$$
Advantages over ReLU:
1. **Smooth**: derivative exists everywhere, including at z=0
2. **Non-monotonic**: has a slight negative lobe for z<0, allowing gradient flow even for weakly-negative inputs
3. **Self-gated**: no separate gating vector needed (unlike GLU)

**SiLU = Swish(1, z):**
$$\text{SiLU}(z) = z \sigma(z) = \frac{z}{1 + e^{-z}}$$

**SwiGLU = SiLU as gate in GLU:**
Replace σ(xV) in GLU with SiLU(xW_gate):
$$\text{SwiGLU} = \text{SiLU}(xW_\text{gate}) \odot (xW_\text{up})$$

Shazeer (2020) showed SwiGLU outperforms other GLU variants (ReLU, GELU, Swish) on translation tasks. Used in PaLM, LLaMA, Mistral, Qwen, Gemma.

**Why SiLU beats ReLU for the gate:**
- ReLU gate: if z < 0, gate = 0 exactly → gradient is zero → neuron never recovers ("dead neuron")
- SiLU gate: if z < 0 slightly, gradient is small but nonzero → neuron can still recover during training
- At inference: SiLU's smooth curve produces richer intermediate representations

## 5. Derivation of the 8/3 Parameter Budget

The goal: use SwiGLU's 3-matrix design but keep the same total parameter count as the original 2-matrix FFN.

**Standard 2-matrix FFN (original Transformer):**

Uses H_int = 4×D for both up and down:
- up:   D → 4D   → 4D² parameters
- down: 4D → D   → 4D² parameters
- **Total: 8D² parameters per layer**

**SwiGLU has a third matrix:**

- gate: D → H_int  → D × H_int parameters
- up:   D → H_int  → D × H_int parameters
- down: H_int → D  → H_int × D parameters
- **Total: 3 × H_int × D parameters**

**Solve for H_int that matches the 8D² budget:**

$$3 \times H_\text{int} \times D = 8D^2 \implies H_\text{int} = \frac{8D}{3}$$

This is where "8/3" comes from: the intermediate dimension needed to make a 3-matrix FFN cost the same as a 2-matrix one.

**For Qwen3-8B (D=4096):**

$$H_\text{int} = \frac{8 \times 4096}{3} \approx 10{,}922$$

Round up to the nearest multiple of 256 → **12,288**. (Tensor cores operate in multiples of 16; 256 gives alignment headroom for efficient tiling.)

Verify: 3 × 12,288 × 4,096 × 2 bytes = 301 MB per layer × 36 layers = **10.8 GB** of MLP weights — about 65% of Qwen3-8B's total parameter storage.

## 6. Symbol Table

| Symbol | Definition | Python / PyTorch | CuTeDSL | CUDA / Triton |
|---|---|---|---|---|
| x | Input (from ln2) | `x: Tensor[B,T,D]` | `mX: cute.Tensor` | `const bf16* x` |
| D | Hidden dimension | `D = 4096` | `D = 4096` | `const int D = 4096` |
| H_int | Intermediate dimension | `H_int = 12288` | `H = 12288` | `const int H = 12288` |
| W_gate | Gate projection weight | `gate_proj.weight: Tensor[H_int,D]` | `mW_gate` | `const bf16* W_gate` |
| W_up | Up projection weight | `up_proj.weight: Tensor[H_int,D]` | `mW_up` | `const bf16* W_up` |
| W_down | Down projection weight | `down_proj.weight: Tensor[D,H_int]` | `mW_down` | `const bf16* W_down` |
| g | Gate activation | `g = x @ W_gate.T` | `mG = GEMM(mX, mW_gate)` | `g[i] = dot(x, W_gate[i])` |
| u | Up activation | `u = x @ W_up.T` | `mU = GEMM(mX, mW_up)` | `u[i] = dot(x, W_up[i])` |
| σ(z) | Sigmoid function | `torch.sigmoid(z)` | `cute.sigmoid(z)` | `1.0f/(1+expf(-z))` |
| SiLU(z) | z·σ(z) | `torch.nn.functional.silu(z)` | `z * cute.sigmoid(z)` | `z/(1+expf(-z))` |
| h | Gated intermediate | `h = silu(g) * u` | `mH[i] = silu(mG[i]) * mU[i]` | `h[i] = silu_mul(g[i], u[i])` |
| y | MLP output | `y = h @ W_down.T` | `mOut = GEMM(mH, mW_down)` | output of down GEMM |
| ⊙ | Element-wise multiply | `*` | `*` | `*` |

In [ ]:
import torch, math

torch.manual_seed(0)
B, T, D, H_int = 1, 4, 4096, 12288
x = torch.randn(B, T, D)

W_gate = torch.randn(H_int, D) * 0.02
W_up   = torch.randn(H_int, D) * 0.02
W_down = torch.randn(D, H_int) * 0.02

# ── Level 0: intent ────────────────────────────────────────────────
# y = swiglu_mlp(x, W_gate, W_up, W_down)

# ── Level 1: three steps ───────────────────────────────────────────
# gate_out = x @ W_gate.T
# up_out   = x @ W_up.T
# y        = (silu(gate_out) * up_out) @ W_down.T

# ── Level 2: expand silu ────────────────────────────────────────────
# silu(z) = z * sigmoid(z) = z / (1 + exp(-z))

# ── Level 3: expand sigmoid ─────────────────────────────────────────
# silu(z) = z / (1 + exp(-z))

# ── Level 4: fully explicit ─────────────────────────────────────────
gate_out = x @ W_gate.T                               # [B,T,H_int]
up_out   = x @ W_up.T                                 # [B,T,H_int]

sigmoid_gate = 1.0 / (1.0 + torch.exp(-gate_out))    # σ(gate)   [B,T,H_int]
silu_gate    = gate_out * sigmoid_gate                 # z·σ(z)    [B,T,H_int]
h            = silu_gate * up_out                      # gated     [B,T,H_int]
y            = h @ W_down.T                            # [B,T,D]

# ── PyTorch shorthand (same result) ────────────────────────────────
y_ref = torch.nn.functional.silu(gate_out) * up_out @ W_down.T
err   = (y - y_ref).abs().max()

print(f"x shape:        {list(x.shape)}")
print(f"gate_out shape: {list(gate_out.shape)}  ({B*T} rows × {H_int} cols)")
print(f"silu_gate range: [{silu_gate.min():.3f}, {silu_gate.max():.3f}]")
print(f"h shape:         {list(h.shape)}")
print(f"y shape:         {list(y.shape)}")
print(f"Max err vs torch: {err:.2e}")
assert err < 1e-4
print("✓ Nested expansion matches torch SiLU × up output")

In [ ]:
import torch

z = torch.linspace(-3, 3, 100)

relu  = torch.relu(z)
silu  = torch.nn.functional.silu(z)
gelu  = torch.nn.functional.gelu(z)
sigma = torch.sigmoid(z)

print("Activation function comparison at key values:")
print(f"{'z':>6} {'ReLU':>8} {'SiLU':>8} {'GELU':>8} | {'SiLU gradient':>14}")
for val in [-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0]:
    z_t = torch.tensor(val, requires_grad=True)
    s   = torch.nn.functional.silu(z_t)
    s.backward()
    grad = z_t.grad.item()
    r = max(val, 0)
    si = torch.nn.functional.silu(torch.tensor(val)).item()
    ge = torch.nn.functional.gelu(torch.tensor(val)).item()
    print(f"{val:>6.1f} {r:>8.4f} {si:>8.4f} {ge:>8.4f} | {grad:>14.4f}")

print()
print("Key insight — at z=-0.5:")
print(f"  ReLU gradient:  0.0000  (dead — no gradient flow)")
print(f"  SiLU gradient:  {torch.autograd.grad(torch.nn.functional.silu(torch.tensor(-0.5, requires_grad=True)), torch.tensor(-0.5, requires_grad=True))[0].item():.4f}  (non-zero — can recover during training)")

## 7. Data Flow at Batch Scale

At B=32 concurrent users, T=512 tokens:
- gate/up each: [32×512, 4096] @ [4096, 12288].T → [16384, 12288]
- FLOPs (gate + up): 2 × 2 × 16384 × 4096 × 12288 = 3.3 TFLOP
- SiLU+mul: 16384 × 12288 × 3 ops = 0.6G FLOPs (trivial)
- down: 2 × 16384 × 12288 × 4096 = 1.65 TFLOP
- Total MLP FLOPs: ~5 TFLOP per forward pass (per batch)

Arithmetic intensity (gate_proj, prefill M=2048):
- FLOPs: 2 × 2048 × 4096 × 12288 = 206 GFLOPs
- Bytes: (2048×4096 + 4096×12288 + 2048×12288) × 2 = 204 MB
- Intensity: 206G/204M ≈ **1010 FLOP/byte** — strongly compute-bound

At decode M=1:
- FLOPs: 2 × 1 × 4096 × 12288 = 100 MFLOPs
- Bytes: 4096×12288×2 ≈ 96 MB (weight reads dominate)
- Intensity: ~1 FLOP/byte — memory-bound (same story as all decode GEMMs)

## 8. Kernel Fusion: silu_and_mul

After the gate and up GEMMs produce two [B·T, H_int] tensors, the next step is:
```
h = silu(gate) * up    # element-wise
```

**Without fusion — 3 separate kernels, 5 HBM passes of [B·T, H_int]:**
```
silu kernel:   reads gate_out (1) → writes silu_gate (1)           = 2 passes
mul kernel:    reads silu_gate (1) + reads up_out (1) → writes h (1) = 3 passes
               ─────────────────────────────────────────────────────────────────
               Total: 5 HBM passes  +  temporary silu_gate allocation in HBM
```

**With fused silu_and_mul — 1 kernel, 3 HBM passes:**
```
one kernel:    reads gate_out (1) + reads up_out (1) → computes silu(gate)*up in registers → writes h (1)
               ─────────────────────────────────────────────────────────────────
               Total: 3 HBM passes  +  no temporary allocation
```

**Saving: 2 fewer HBM passes of [B·T, H_int].** At prefill (B=1, T=2048, H_int=12288, BF16): each pass = 2048×12288×2 ≈ 48 MB. Two passes saved = ~96 MB of avoided HBM traffic per layer.

**Bigger win: fuse gate+up into one GEMM:**

Instead of two separate GEMMs (gate, up), combine the weights:
```python
W_combined = torch.cat([W_gate, W_up], dim=0)   # shape [2×H_int, D] = [24576, 4096]
gate_up    = x @ W_combined.T                    # one GEMM → [B·T, 24576]
gate, up   = gate_up.chunk(2, dim=-1)            # slice in-place
```
One GEMM instead of two → 2× better SM utilization at decode, better L2 cache behavior for the weight tiles. Then apply silu_and_mul in the same epilogue. This is the pattern FlashInfer, vLLM, and TRT-LLM all use.

## 9. Production Implementations

| Library | API | What it does |
|---|---|---|
| FlashInfer | `flashinfer.activation.silu_and_mul(out, input)` | Fused SiLU + element-wise multiply, writes result to out in-place |
| vLLM | `csrc/activation_kernels.cu::silu_and_mul_kernel` | Same pattern; launched after gate+up GEMMs |
| vLLM | `csrc/ops/gemm_kernels.cu` + Marlin | W4A16 quantized MLP GEMMs |
| SGLang | Calls FlashInfer `silu_and_mul` | `srt/layers/activation.py` |
| TRT-LLM | MLP plugin compiles gate+up+silu_and_mul+down into one TRT engine | Full FFN sublayer as single TRT op |
| quack (this project) | Not yet written | Target: fuse gate+up GEMM + silu_and_mul + potentially down GEMM epilogue |

**Common fusion pattern across all implementations:**
1. Fused QKV projection (Q+K+V in one GEMM)
2. Fused gate+up projection (optional: one combined [2H_int, D] GEMM)
3. Fused silu_and_mul (one kernel reads gate+up, writes h)
4. Down projection (standard GEMM, sometimes with fused RMSNorm epilogue from next layer)

In [ ]:
# The silu_and_mul operation: h[i] = silu(gate[i]) * up[i]
# This is an element-wise kernel — no reduction, no cross-thread communication.
# Thread i handles element i (or a chunk of elements).

# @cute.kernel
# def silu_and_mul(mGate, mUp, mOut):
#     # One thread per element (or vectorized: one thread per 4 elements)
#     idx = cute.thread_idx('x') + cute.block_idx('x') * cute.block_dim('x')
#
#     if idx < mGate.shape[0]:    # bounds check
#         g = cutlass.Float32(mGate[idx])   # BF16 → FP32
#         u = cutlass.Float32(mUp[idx])     # BF16 → FP32
#
#         # SiLU: g * sigmoid(g) = g / (1 + exp(-g))
#         sigmoid_g = 1.0 / (1.0 + cute.exp(-g))
#         silu_g    = g * sigmoid_g
#
#         mOut[idx] = cutlass.BFloat16(silu_g * u)   # FP32 → BF16

print("silu_and_mul kernel properties:")
print("  Fully independent threads — no reduction, no SMEM, no barriers")
print("  2 BF16 reads (gate, up) + 1 BF16 write + ~6 FLOPs = 1 FLOP/byte")
print("  Always memory-bound (trivially fast regardless of fusion)")
print()
print("CuTeDSL primitives:")
print("  cutlass.Float32(bf16)  → upcast BF16 to FP32 for accumulation")
print("  cute.exp(-g)           → hardware __expf(-g), ~1 cycle")
print("  cutlass.BFloat16(f32)  → downcast result back to BF16")
print()
print("Vectorization: load 4 BF16 values at once (float4 load = 128-bit)")
print("  mGate.load_v4(idx) → 4 values in one instruction")
print("  4× throughput improvement vs scalar loads")

## Key Takeaways

- **What SwiGLU does:** Adds a learned "gate" to the FFN. Instead of up→ReLU→down, it computes SiLU(gate) ⊙ up → down. The gate is a second projection that learns which intermediate features to amplify and which to suppress — it acts like a soft filter that shapes the information flowing through the up projection.

- **Why SiLU instead of ReLU:** ReLU sets any negative gate value to exactly zero, permanently blocking gradient flow to that neuron ("dead neuron problem"). SiLU has a small but non-zero gradient even for negative inputs — a neuron that's currently being suppressed can still receive a gradient signal and potentially reactivate. Shazeer (2020) showed this produces measurably better models on translation benchmarks.

- **The cost and where to optimize:** SwiGLU needs 3 weight matrices instead of 2. Setting H_int = 8D/3 ≈ 12,288 matches the original 2-matrix parameter budget. The MLP block is ~65% of all Qwen3-8B parameters and ~50% of all FLOPs. The two highest-ROI kernel optimizations are: fusing gate+up into one combined GEMM (halves GEMM launches at decode), and fusing silu_and_mul (saves 2 HBM passes of [B·T, 12288] per layer).